## Lab 2: Tokenisierung und Positionskodierungen (Positional Embeddings)
**Gruppe:** Phillip Graf, Konstantin Schmidt, Fabian Holländer

**Ersteller:** Phillip Graf

In diesem Lab importieren wir unseren Datensatz und führen die notwendigen Vorarbeitungsschritte durch. Anschließend konzentrieren wir uns auf die Generierung von Token aus den Rezeptdaten und implementieren als letzten Schritt Positionskodierungen, um die Sequenzen für das Modell vorzubereiten.

### Quellen

Die originalen Datenquellen bitte dem [lab1.ipynb](./lab1.ipynb) entnehmen.

Für Lern- und Testzwecke arbeiten wir zunächst mit einem reduzierten Datensatz, der 100.000 annotierte Rezepte enthält.

> **Reduzierter Datensatz (100k Rezepte):**
> [reduced_dataset_100k.csv herunterladen](https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv)

#### Optionaler vollständiger Datensatz
Für ein umfassenderes Training auf größerer Skala steht zusätzlich der vollständige Datensatz mit insgesamt 2 Millionen annotierten Rezepten zur Verfügung:

> **Vollständiger Datensatz (2M Rezepte):**
> [full_dataset.csv herunterladen](https://syncandshare.lrz.de/dl/finR5gtyQx3FNL5P2hz6H/full_dataset.csv)


Die technische Grundlage dieses Lab ist Kapitel 2 "Working with Text Data" aus Sebastian Raschkas Repository **LLMs from scratch**. Die dort gezeigten Konzepte zu Tokenisierung, Token-IDs, Sliding-Window-Dataset, DataLoader, Token-Embeddings und Positional Embeddings wurden auf unseren Rezeptdatensatz übertragen: [rasbt/LLMs-from-scratch, ch02](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02).

Für Byte-Pair-Encoding (BPE) verwenden wir die OpenAI-Bibliothek `tiktoken`. OpenAI beschreibt `tiktoken` als BPE-Tokenizer für OpenAI-Modelle und zeigt im Cookbook, wie Encodings wie `cl100k_base` geladen, Texte encodiert/decodiert und verschiedene Encodings verglichen werden: [OpenAI Cookbook: How to count tokens with tiktoken](https://developers.openai.com/cookbook/examples/how_to_count_tokens_with_tiktoken), [openai/tiktoken](https://github.com/openai/tiktoken).

## Notwendige Bibliotheken importieren


In [159]:
import pandas as pd
import re
import tiktoken
import torch
from IPython.display import display
import os

## Den Datensatz laden oder lokalen Raw-Text verwenden

Damit die CSV-Datei nicht bei jedem Notebook-Run erneut aus der Cloud geladen werden muss, speichern wir den fertig formatierten `raw_text` lokal unter `datasets/pre-training/raw_text.txt`. Wenn diese Datei existiert und nicht leer ist, wird sie direkt geladen. Nur beim ersten Lauf oder nach dem Löschen der Datei wird der CSV-Datensatz erneut geladen und daraus `raw_text` erzeugt.


In [160]:
RAW_TEXT_PATH = "datasets/pre-training/raw_text.txt"
os.makedirs("datasets/pre-training", exist_ok=True)

REDUCED_DATASET_URL = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv"
FULL_DATASET_URL = "https://syncandshare.lrz.de/dl/finR5gtyQx3FNL5P2hz6H/full_dataset.csv"

USE_FULL_DATASET = False
DEMO_ROW_LIMIT = 100_000

if os.path.exists(RAW_TEXT_PATH) and os.path.getsize(RAW_TEXT_PATH) > 0:
    with open(RAW_TEXT_PATH, "r", encoding="utf-8") as f:
        raw_text = f.read()
    df = None
    full_raw_text = raw_text
    print(f"Raw text aus lokalem Cache geladen: {RAW_TEXT_PATH}")
    print(f"Geladene Zeichen: {len(raw_text):,}")
else:
    cloud_url = FULL_DATASET_URL if USE_FULL_DATASET else REDUCED_DATASET_URL

    try:
        print(f"Kein lokaler Raw-Text gefunden. Versuche Cloud-Download: {cloud_url}")
        df = pd.read_csv(cloud_url, nrows=DEMO_ROW_LIMIT)
    except Exception as error:
        raise RuntimeError(
            "Datensatz konnte nicht aus der Cloud geladen werden. "
            "Bitte URL prüfen oder Internetverbindung herstellen."
        ) from error

    required_columns = {"title", "ingredients", "directions"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Im Datensatz fehlen erwartete Spalten: {sorted(missing_columns)}")

    print(f"Geladene Zeilen: {len(df):,}")
    print(f"Spalten: {list(df.columns)}")
    display(df.head())


Raw text aus lokalem Cache geladen: datasets/pre-training/raw_text.txt
Geladene Zeichen: 1,000,000


## Formatierung des Datensatzes zu einem einzigen Eingabestring
Dies ermöglicht es uns, das Vokabular zu deduplizieren, konsistente Token zu generieren und die Sequenziteration später im Prozess erheblich zu rationalisieren.

Da nur die drei Spalten **title** (Titel), **ingredients** (Zutaten) und **directions** (Zubereitungsschritte) für das Textverständnis notwendig sind, verwenden wir nur diese, um unseren zusammenhängenden Eingabestring zu erzeugen.

In [161]:
def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recipe: {title}\nIngredients: {ingredients}\nDirections: {directions}"

if df is not None:
    full_raw_text = "".join(df.apply(format_csv, axis=1))

    # Das Limit gilt nur für dieses Demonstrationsnotebook, damit die Zellen schnell laden.
    # Für echte Trainingslaeufe kann der vollstaendige Text oder ein groesserer Ausschnitt genutzt werden.
    DEMO_CHAR_LIMIT = 1_000_000
    raw_text = full_raw_text[:DEMO_CHAR_LIMIT]

    os.makedirs("datasets/pre-training", exist_ok=True)
    with open(RAW_TEXT_PATH, "w", encoding="utf-8") as f:
        f.write(raw_text)
    print(f"Raw text gespeichert unter: {RAW_TEXT_PATH}")
else:
    print(f"Verwende bereits geladenen Raw-Text aus: {RAW_TEXT_PATH}")

if __name__ == '__main__':
    print("Der Datensatz steht als zusammenhaengender String zur Verfügung.")
    print(f"Zeichen im zusammengesetzten Text: {len(full_raw_text):,}")
    print(f"Für dieses Notebook genutzte Zeichen: {len(raw_text):,}")
    print("\nVorschau der ersten 300 Zeichen:")
    print(raw_text[:300])


Verwende bereits geladenen Raw-Text aus: datasets/pre-training/raw_text.txt
Der Datensatz steht als zusammenhaengender String zur Verfügung.
Zeichen im zusammengesetzten Text: 1,000,000
Für dieses Notebook genutzte Zeichen: 1,000,000

Vorschau der ersten 300 Zeichen:
Recipe: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
Directions: In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and b


## 2.1 Erstellen eines einfachen Tokenizers

In diesem Schritt entwickeln wir einen unkomplizierten Tokenizer, um den zusammengehängten Datensatz zu verarbeiten:
- Der Eingabestring wird an Leerzeichen und bestimmten Satzzeichen gespaltet, die als Worttrenner dienen, einschließlich: , . : ; ? _ ! " ( )
- Dieser Prozess führt zu einer Liste von Token. In diesem Kontext kann ein Token ein vollständiges Wort, ein einzelnes Zeichen oder eine Sequenz von Spezialsymbolen (z. B. @#) sein.
---

**Beispiel**

`This is a sample text: with numbers 123, special characters !@# and a few more. .`

Der Tokenizer identifiziert einzelne Wörter wie „text“, bewahrt aber auch zusammenhängende Spezialzeichen wie „@#“ als eigenständige Token.

In [162]:
# the tokenizer function (basically just a split function)
def split_text(text):
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    tokens = [token for token in tokens if token.strip()]
    return tokens

In [163]:
# Example
example_text = "This is a sample text: with numbers 123, special characters !@# and a few more.  ."
result = split_text(example_text)
if __name__ == '__main__':
    print(result)

['This', 'is', 'a', 'sample', 'text', ':', 'with', 'numbers', '123', ',', 'special', 'characters', '!', '@#', 'and', 'a', 'few', 'more', '.', '.']


In [164]:
# Tokenizer on our dataset
split = split_text(raw_text)

if __name__ == '__main__':
    print(f"Anzahl der Tokens im Demo-Subset: {len(split):,}")
    print(f"Erste 40 Tokens: {split[:40]}")
    print(f"Letzte 10 Tokens: {split[-10:]}")

Anzahl der Tokens im Demo-Subset: 234,635
Erste 40 Tokens: ['Recipe', ':', 'No-Bake', 'Nut', 'Cookies', 'Ingredients', ':', '1', 'c', '.', 'firmly', 'packed', 'brown', 'sugar', ',', '1/2', 'c', '.', 'evaporated', 'milk', ',', '1/2', 'tsp', '.', 'vanilla', ',', '1/2', 'c', '.', 'broken', 'nuts', '(', 'pecans', ')', ',', '2', 'Tbsp', '.', 'butter', 'or']
Letzte 10 Tokens: ['time', 'approximately', '45', 'minutes', '.', 'Recipe', ':', 'Apple', 'Nut', 'Ring']


## 2.2 Umwandlung von Token in Token-IDs
Dies ist ein entscheidender Vorbereitungsschritt für das neuronale Netzwerk, da das Modell nur numerische Werte verarbeiten kann.
In der folgenden Zelle sehen Sie eine Funktion und deren Ausgabe, die demonstriert, wie ein Token einer ID zugewiesen wird. Der Prozess ist unkompliziert: Dem ersten Token in der sortierten Vokabularliste wird die ID 0 zugewiesen, dem zweiten Token die ID 1 und so weiter.

Dies gibt das **Vokabular** (Vocab) aus, also die Zuordnung zwischen Token und Token-ID. Wir sortieren die Token vor der ID-Vergabe, damit die Zuordnung bei jedem Notebook-Durchlauf deterministisch bleibt.

In [165]:
all_words = sorted(set(split))
vocab = {token: integer for integer, token in enumerate(all_words)}

print(f"Vokabulargroesse: {len(vocab):,}")
print("Erste 20 Vokabular-Eintraege:")
for item in list(vocab.items())[:20]:
    print(item)

Vokabulargroesse: 5,086
Erste 20 Vokabular-Eintraege:
('!', 0)
('"', 1)
('#1', 2)
('#2', 3)
('%', 4)
('&', 5)
("'", 6)
('(', 7)
(')', 8)
('*', 9)
('*Apricot', 10)
('*Can', 11)
('*Chocolate', 12)
('*Cool', 13)
('*I', 14)
('*Note', 15)
('*Ranch', 16)
('+', 17)
('+/-', 18)
('+cooking', 19)


## 2.3 Aufbau eines gesamten Tokenizer-Systems
Die folgende Zelle kombiniert diese Komponenten zu einer Tokenizer-Klasse. Diese Klasse nimmt Text als Eingabe, spaltet ihn in Token auf und weist diesen Token IDs zu, wie wir es in den obigen Zellen gesehen haben. Darüber hinaus enthält sie eine Dekodierungsfunktion, um Token-IDs wieder in Strings umzuwandeln. Dieser Dekodierungsschritt ist für uns unerlässlich, um die numerische Ausgabe des Modells wieder in von Menschen lesbaren Text zu übersetzen.

In [166]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [167]:
# Beispiel für die Verwendung des Tokenizers
tokenizer = SimpleTokenizerV1(vocab)
ids = tokenizer.encode(raw_text)
decoded_ids = tokenizer.decode(ids)

if __name__ == '__main__':
    print(f"Anzahl Token-IDs: {len(ids):,}")
    print(f"Erste 30 Token-IDs: {ids[:30]}")
    print("\nDekodierter Ausschnitt:")
    print(decoded_ids[:300])

Anzahl Token-IDs: 234,635
Erste 30 Token-IDs: [1716, 247, 1454, 1469, 655, 1131, 247, 27, 2643, 24, 3209, 3946, 2596, 4691, 20, 40, 2643, 24, 3147, 3756, 20, 40, 4875, 24, 4946, 20, 40, 2643, 24, 2592]

Dekodierter Ausschnitt:
Recipe : No-Bake Nut Cookies Ingredients : 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts( pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits Directions : In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk an


## 2.4 Spezielle Token hinzufügen
Im nächsten Schritt integrieren wir spezielle Token in unseren Tokenizer, um spezifische strukturelle und vokabularische Anforderungen zu erfüllen:
- <|unk|> (Unbekannt-Token): Dieses Token repräsentiert Wörter oder Symbole, die im Trainingsvokabular nicht vorhanden waren. Anstatt einen Fehler auszugeben, wenn während der Kodierung ein unbekanntes Wort auftritt, weist das System dieses einfach der <|unk|>-ID zu.
- <|endoftext|> (Textende-Token): Dies dient dem Modell als Signal, dass der aktuelle Satz oder die aktuelle Phrase beendet ist.
---
**Beispiel**

Mit dem Vokabular unseres (reduzierten) Datensatzes (siehe oben) und dem folgenden Eingabebeispiel:

``Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.``

Erhalten wir diese kodierte Liste von Token-IDs:

``[19, 0, 19, 19, 19, 19, 19, 18, 19, 19, 19, 19, 19, 19, 19, 1]``

Wenn wir diese Token-IDs dekodieren, erhalten wir im Gegenzug die folgenden Token zurück:

``<|unk|>, <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|endoftext|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|>.``

Das bedeutet, dass all diese Wörter unserem Vokabular leider unbekannt sind. Lediglich die Satzzeichen „,“ und „.“ werden als bekannte Token erkannt.

In [168]:
# Adjusted tokenizer with unknown-token handling
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [169]:
# add the new special tokens to the vocabulary
all_tokens = sorted(list(set(split)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [170]:
# Test the new tokenizer with the updated vocabulary on an example input text
tokenizer = SimpleTokenizerV2(vocab)

# Example sentences
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

# Get the token IDs for the example text
token_ids = tokenizer.encode(text)

decoded_tokens = tokenizer.decode(token_ids)

if __name__ == '__main__':
    print(f"Example text: \n{text}\n")
    print(f"Token IDs: {token_ids}\n")
    print(f"Decoded text: \n{decoded_tokens}\n")

Example text: 
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

Token IDs: [1080, 20, 3043, 5078, 3628, 4749, 249, 5086, 1126, 4777, 5087, 5087, 3869, 4777, 5087, 24]

Decoded text: 
Hello, do you like tea? <|endoftext|> In the <|unk|> <|unk|> of the <|unk|>.



## 2.5 Byte-Pair-Encoding

Unser aktueller Tokenizer erkennt nur exakte Wörter. Indem wir unbekannte Begriffe in Sub-Token zerlegen, anstatt sie einfach als „unbekannt“ zu kennzeichnen, stellen wir sicher, dass das Modell neue Wörter immer noch verarbeiten kann, solange ihre einzelnen Bestandteile Teil des Vokabulars sind.

Anstatt dies selbst zu bauen, verwenden wir die `tiktoken`-Bibliothek von OpenAI mit der `cl100k_base`-Kodierung. Im Vergleich zur älteren `gpt2`-Kodierung aus dem Originalkapitel segmentiert `cl100k_base` unseren Beispieltext anders und kann insbesondere kompakte oder sonderzeichenreiche Zeichenfolgen in kleinere Tokenbestandteile zerlegen. Mehr Information dazu findet sich im OpenAI Cookbook: [How to count tokens with tiktoken](https://developers.openai.com/cookbook/examples/how_to_count_tokens_with_tiktoken).

---
**Beispiel**

Das folgende Beispiel demonstriert die Auswirkung von Byte-Pair-Encoding (BPE) im Vergleich zu unserem SimpleTokenizer. Beobachten Sie, wie sich die Anzahl der Token verändert, wenn Leerzeichen entfernt werden.

Standardtext:

``Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.``

Komprimierter Text ohne Leerzeichen:

``Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace.``

- SimpleTokenizerV2:
    - Standardtext: Generiert 16 Token.
    - Komprimierter Text (keine Leerzeichen): Generiert nur 6 Token. Er schafft es nicht, einzelne Wörter ohne Leerzeichen zu erkennen.

- Tiktoken (OpenAI):
    - Standardtext: Generiert 19 Token.
    - Komprimierter Text (keine Leerzeichen): Generiert 22 Token. Es identifiziert Sub-Wort-Einheiten und behält auch ohne Leerzeichen mehr Struktur bei.

Danach prüfen wir denselben Effekt direkt an einem Rezeptausschnitt mit Mengenangaben, Bindestrich, Abkürzungen und Sonderzeichen.

In [171]:
tokenizer = SimpleTokenizerV2(vocab)

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.")

# Get the token IDs for the example text
token_ids = tokenizer.encode(text)

decoded_tokens = tokenizer.decode(token_ids)

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

if __name__ == '__main__':
    print(f"Token IDs: {token_ids}\n")
    print(f"Token Count: {len(token_ids)}")
    print(f"Decoded text: \n{decoded_tokens}\n")
    # Print the list
    print("List of individual tokens:")
    print(decoded_list)


Token IDs: [1080, 20, 3043, 5078, 3628, 4749, 249, 5086, 1126, 4777, 5087, 5087, 3869, 4777, 5087, 24]

Token Count: 16
Decoded text: 
Hello, do you like tea? <|endoftext|> In the <|unk|> <|unk|> of the <|unk|>.

List of individual tokens:
['Hello', ',', 'do', 'you', 'like', 'tea', '?', '<|endoftext|>', 'In', 'the', '<|unk|>', '<|unk|>', 'of', 'the', '<|unk|>', '.']


In [172]:
text = ( "Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace." )

# Get the token IDs for the example text
token_ids = tokenizer.encode(text)
print(f"Token IDs: {token_ids}\n")
print(f"Token Count: {len(token_ids)}")

decoded_tokens = tokenizer.decode(token_ids)
print(f"Decoded text: \n{decoded_tokens}\n")

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

# Print the list
print("List of individual tokens:")
print(decoded_list)


Token IDs: [1080, 20, 5087, 249, 5087, 24]

Token Count: 6
Decoded text: 
Hello, <|unk|>? <|unk|>.

List of individual tokens:
['Hello', ',', '<|unk|>', '?', '<|unk|>', '.']


In [173]:
tokenizer = tiktoken.get_encoding("cl100k_base")

# Example text with the byte-pair-encoding
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.")
token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(f"Token IDs: {token_ids}")
print(f"Token Count: {len(token_ids)}")

decoded_tokens = tokenizer.decode(token_ids)
print(f"\nDecoded Tokens: \n{decoded_tokens}\n")

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

# Print the list
print("List of individual tokens:")
print(decoded_list)


Token IDs: [9906, 11, 656, 499, 1093, 15600, 30, 220, 100257, 763, 279, 7160, 32735, 7317, 2492, 315, 279, 44439, 13]
Token Count: 19

Decoded Tokens: 
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

List of individual tokens:
['Hello', ',', ' do', ' you', ' like', ' tea', '?', ' ', '<|endoftext|>', ' In', ' the', ' sun', 'lit', ' terr', 'aces', ' of', ' the', ' palace', '.']


In [174]:
# Example text but without spaces to see the effect of the byte-pair-encoding
text = ( "Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace." )
token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(f"Token IDs: {token_ids}")
print(f"Token Count: {len(token_ids)}")

decoded_tokens = tokenizer.decode(token_ids)
print(f"\nDecoded Tokens: \n{decoded_tokens}\n")

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

# Print the list
print("List of individual tokens:")
print(decoded_list)

Token IDs: [9906, 12260, 2303, 283, 7792, 7870, 64, 30, 100257, 644, 6509, 359, 75, 3328, 81, 2492, 1073, 339, 752, 278, 580, 13]
Token Count: 22

Decoded Tokens: 
Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace.

List of individual tokens:
['Hello', ',d', 'oy', 'ou', 'lik', 'ete', 'a', '?', '<|endoftext|>', 'In', 'thes', 'un', 'l', 'itter', 'r', 'aces', 'of', 'th', 'ep', 'al', 'ace', '.']


In [175]:
# Generate a comparison table for the two tokenizers on the two example texts

# Initialize tokenizers
tokenizer_v2 = SimpleTokenizerV2(vocab)
tokenizer_tik = tiktoken.get_encoding("cl100k_base")

# Define input strings
text_standard = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
text_compressed = "Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace."

results = []

# Process strings
for label, text in [("Standard text (with spaces)", text_standard), ("Compressed text (no spaces)", text_compressed)]:
    # SimpleTokenizerV2
    ids_v2 = tokenizer_v2.encode(text)
    
    # Tiktoken (BPE)
    ids_tik = tokenizer_tik.encode(text, allowed_special={"<|endoftext|>"})
    
    results.append({
        "label": label,
        "v2_count": len(ids_v2),
        "tik_count": len(ids_tik)
    })

# Print comparison table
print(f"{'Text Version':<30} | {'SimpleV2 Count':<15} | {'Tiktoken Count':<15}")
print("-" * 60)
for res in results:
    print(f"{res['label']:<30} | {res['v2_count']:<15} | {res['tik_count']:<15}")

Text Version                   | SimpleV2 Count  | Tiktoken Count 
------------------------------------------------------------
Standard text (with spaces)    | 16              | 19             
Compressed text (no spaces)    | 6               | 22             


In [176]:
# BPE-Vergleich an einem Rezeptausschnitt mit typischen Sonderfällen
recipe_example = (
    "Recipe: No_Bake Nut Cookies\n"
    "Ingredients: 1/2 c. evaporated milk, 2 Tbsp. butter, bake at 275° for 3 hours."
)

simple_recipe_ids = tokenizer_v2.encode(recipe_example)
bpe_recipe_ids = tokenizer_tik.encode(recipe_example, allowed_special={"<|endoftext|>"})

recipe_token_comparison = pd.DataFrame([
    {
        "Tokenizer": "SimpleTokenizerV2",
        "Token Count": len(simple_recipe_ids),
        "Erste Tokens": [tokenizer_v2.decode([token_id]) for token_id in simple_recipe_ids[:20]],
    },
    {
        "Tokenizer": "tiktoken cl100k_base",
        "Token Count": len(bpe_recipe_ids),
        "Erste Tokens": [tokenizer_tik.decode([token_id]) for token_id in bpe_recipe_ids[:20]],
    },
])

print(recipe_example)
display(recipe_token_comparison)

Recipe: No_Bake Nut Cookies
Ingredients: 1/2 c. evaporated milk, 2 Tbsp. butter, bake at 275° for 3 hours.


,Tokenizer,Token Count,Erste Tokens
0,SimpleTokenizerV2,27,"[Recipe, :, No, <|unk|>, Bake, Nut, Cookies, I..."
1,tiktoken cl100k_base,37,"[Recipe, :, No, _B, ake, Nut, Cookies, \n, ..."


### Beobachtungen am Rezepttext

Rezeptdaten enthalten viele Tokens, die in normalem Fließtext seltener vorkommen: Mengenangaben wie `1/2`, Abkürzungen wie `c.` oder `Tbsp.`, Bindestrich-Wörter wie `No-Bake`, Zahlen mit Einheiten und Sonderzeichen wie `°`. Der einfache Tokenizer funktioniert nur zuverlässig, wenn genau dieselben Schreibweisen bereits im Vokabular vorkommen. Neue oder leicht abweichende Schreibweisen wie `No_Bake` werden deshalb schnell zu `<|unk|>`.

BPE ist für diese Daten robuster, weil unbekannte Wörter nicht komplett verloren gehen, sondern in kleinere Subword- oder Zeichenbestandteile zerlegt werden. Dadurch bleiben auch bei neuen Rezeptnamen, zusammengesetzten Zutaten oder kompakten Mengenangaben noch verwertbare Informationen erhalten.

## 2.6 Datensampling mit einem Sliding Window (gleitendes Fenster)

Sobald der Text tokenisiert ist, benötigen wir einen Mechanismus, um ihn in das neuronale Netzwerk einzuspeisen. Da ein Modell lernt, Token sequentiell vorherzusagen, trainieren wir es so, dass es eine Sequenz vorheriger Wörter betrachtet, um das nächste vorherzusagen. Um zu verhindern, dass das Modell „schummelt“, sollte es nur den Kontext sehen, der dem Ziel-Token vorausgeht.

Um dies umzusetzen, verwenden wir den Sliding-Window-Ansatz:
- **Eingabesequenz (Input Sequence):** Ein Fenster mit fester Länge aus Token des Textes.
- **Zielsequenz (Target Sequence):** Genau dasselbe Fenster, jedoch um eine Position nach rechts verschoben.

### DataLoader
Um unseren Text effizient in das Modell einzuspeisen, verpacken wir die Sliding-Window-Logik in ein PyTorch-Dataset und einen DataLoader. Dies ermöglicht es uns, große Datenmengen zu verarbeiten, Sequenzen zu mischen und sie in Batches zu organisieren.

**Wichtige Parameter**

``max_length``: Die Größe des Kontextfensters. Es bestimmt, wie viele Token das Modell betrachtet, um das nächste vorherzusagen.

``stride``: Die Distanz, um die sich das Fenster für das nächste Sample verschiebt. Ein Stride, der kleiner als `max_length` ist, erzeugt überlappende Sequenzen. Das liefert mehr Trainingssamples, erhöht aber auch Redundanz.

``batch_size``: Die Anzahl der Sequenzen, die gleichzeitig in einem Trainingsschritt verarbeitet werden.

``shuffle``: Wenn True, wird die Reihenfolge der Sequenzen zufällig bestimmt, damit das Modell nicht nur die Reihenfolge des Quelltexts sieht.

Die folgenden Zellen zeigen zuerst einzelne Sliding-Window-Beispiele und danach eine Vergleichstabelle für mehrere `max_length`-/`stride`-Kombinationen.

In [177]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the text into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [178]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("cl100k_base")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [179]:
def decode_batch(batch, tokenizer):
    inputs, targets = batch
    
    # Extract IDs for the first sequence in the batch
    input_ids = inputs[0].tolist()
    target_ids = targets[0].tolist()
    
    # Decode the full text
    input_text = tokenizer.decode(input_ids)
    target_text = tokenizer.decode(target_ids)
    
    # Decode tokens individually to see the fragments
    input_tokens = [tokenizer.decode([t_id]) for t_id in input_ids]
    target_tokens = [tokenizer.decode([t_id]) for t_id in target_ids]

    print(f"Input batch (Full):   {input_text}")
    print(f"Input tokens (List):   {input_tokens}")
    print("-" * 30)
    print(f"Target batch (Full):  {target_text}")
    print(f"Target tokens (List):  {target_tokens}")

In [180]:
tokenizer = tiktoken.get_encoding("cl100k_base")
enc_text = tokenizer.encode(raw_text, allowed_special={"<|endoftext|>"})

print(f"Raw-text-Laenge: {len(raw_text):,} Zeichen")
print(f"Token-Anzahl mit cl100k_base: {len(enc_text):,}")
print("\nRaw-text-Vorschau:")
print(raw_text[:300])
print(f"\nErste 30 Token-IDs: {enc_text[:30]}")
print(f"Erste 30 dekodierte Tokens: {[tokenizer.decode([token_id]) for token_id in enc_text[:30]]}")

Raw-text-Laenge: 1,000,000 Zeichen
Token-Anzahl mit cl100k_base: 285,289

Raw-text-Vorschau:
Recipe: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
Directions: In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and b

Erste 30 Token-IDs: [29880, 25, 2360, 7826, 731, 18878, 27085, 198, 46847, 25, 220, 16, 272, 13, 32620, 19937, 14198, 13465, 11, 220, 16, 14, 17, 272, 13, 60150, 660, 14403, 11, 220]
Erste 30 dekodierte Tokens: ['Recipe', ':', ' No', '-B', 'ake', ' Nut', ' Cookies', '\n', 'Ingredients', ':', ' ', '1', ' c', '.', ' firmly', ' packed', ' brown', ' sugar', ',', ' ', '1', '/', '2', ' c', '.', ' evapor', 'ated', ' milk', ',', ' ']


In [181]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

# Textprüfung
decode_batch(first_batch, tokenizer)

[tensor([[29880,    25,  2360,  7826]]), tensor([[  25, 2360, 7826,  731]])]
Input batch (Full):   Recipe: No-B
Input tokens (List):   ['Recipe', ':', ' No', '-B']
------------------------------
Target batch (Full):  : No-Bake
Target tokens (List):  [':', ' No', '-B', 'ake']


In [182]:
second_batch = next(data_iter)
print(second_batch)

# Textpruefung
decode_batch(second_batch, tokenizer)

[tensor([[  25, 2360, 7826,  731]]), tensor([[ 2360,  7826,   731, 18878]])]
Input batch (Full):   : No-Bake
Input tokens (List):   [':', ' No', '-B', 'ake']
------------------------------
Target batch (Full):   No-Bake Nut
Target tokens (List):  [' No', '-B', 'ake', ' Nut']


In [183]:
# Kurzes Beispiel, wie der DataLoader mit Batches funktioniert
dataloader = create_dataloader_v1(raw_text, batch_size=3, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[29880,    25,  2360,  7826],
        [  731, 18878, 27085,   198],
        [46847,    25,   220,    16]])

Targets:
 tensor([[   25,  2360,  7826,   731],
        [18878, 27085,   198, 46847],
        [   25,   220,    16,   272]])


In [184]:
# Vergleich mehrerer Context-/Stride-Konfigurationen
sampling_tokenizer = tiktoken.get_encoding("cl100k_base")
sampling_token_ids = sampling_tokenizer.encode(raw_text, allowed_special={"<|endoftext|>"})

sampling_configs = [
    {"max_length": 4, "stride": 1},
    {"max_length": 4, "stride": 4},
    {"max_length": 8, "stride": 4},
    {"max_length": 16, "stride": 8},
    {"max_length": 32, "stride": 32},
]

sampling_rows = []
for config in sampling_configs:
    max_length = config["max_length"]
    stride = config["stride"]
    sample_count = len(range(0, len(sampling_token_ids) - max_length, stride))
    input_ids = sampling_token_ids[:max_length]
    target_ids = sampling_token_ids[1:max_length + 1]

    sampling_rows.append({
        "max_length": max_length,
        "stride": stride,
        "Overlap Tokens": max(max_length - stride, 0),
        "Samples im Demo-Subset": sample_count,
        "Erster Input": sampling_tokenizer.decode(input_ids).replace("\n", "\\n")[:80],
        "Erstes Target": sampling_tokenizer.decode(target_ids).replace("\n", "\\n")[:80],
    })

display(pd.DataFrame(sampling_rows))

,max_length,stride,Overlap Tokens,Samples im Demo-Subset,Erster Input,Erstes Target
0,4,1,3,285285,Recipe: No-B,: No-Bake
1,4,4,0,71322,Recipe: No-B,: No-Bake
2,8,4,4,71321,Recipe: No-Bake Nut Cookies\n,: No-Bake Nut Cookies\nIngredients
3,16,8,8,35660,Recipe: No-Bake Nut Cookies\nIngredients: 1 c....,: No-Bake Nut Cookies\nIngredients: 1 c. firml...
4,32,32,0,8915,Recipe: No-Bake Nut Cookies\nIngredients: 1 c....,: No-Bake Nut Cookies\nIngredients: 1 c. firml...


### Beobachtungen zu `max_length` und `stride`

Ein kleiner `stride` erzeugt viele stark überlappende Samples. Das kann für kleine Datenausschnitte hilfreich sein, weil mehr Trainingsbeispiele entstehen, erhöht aber die Redundanz zwischen den Samples. Wenn `stride == max_length` ist, überlappen die Fenster nicht mehr; dadurch entstehen weniger Samples, aber die Trainingsdaten sind weniger redundant.

Eine größere `max_length` gibt dem Modell mehr Kontext pro Beispiel. Gleichzeitig sinkt bei gleichem Textausschnitt die Anzahl der möglichen Samples, und spätere Trainingsschritte werden speicherintensiver.

## 2.7 Token-Embeddings (Token-Einbettungen)

Token-Embeddings transformieren diskrete Ganzzahl-IDs (Integers) in kontinuierliche Vektoren fester Größe. Anstatt Wörter als willkürliche Zahlen zu behandeln, repräsentieren wir sie in einem hochdimensionalen Raum, in dem das Modell Beziehungen zwischen ihnen lernen kann.

Eine Embedding-Schicht ist im Wesentlichen eine Nachschlagetabelle (Lookup-Table). Anstatt aufwendige Matrixmultiplikationen mit One-Hot-kodierten Vektoren durchzuführen, ruft sie direkt die entsprechende Zeile aus einer Gewichtsmatrix ab. Im Laufe der Zeit passt das Modell diese Vektoren so an, dass Wörter, die in ähnlichen Kontexten verwendet werden, im Vektorraum näher beieinander liegen.

### Zum eigenen Verständnis

**Warum nicht einfach IDs verwenden?**

Obwohl ein Tokenizer jedem Wort eine eindeutige ID (Ganzzahl) zuweist, ist die direkte Verwendung dieser IDs für das Training problematisch:
- Willkürliche Distanz: Liegt das Token 500 in seiner Bedeutung näher an 501 als an 10? Nein. In Ganzzahlform sind die Werte willkürlich und vermitteln keine semantische Beziehung.
- Lineare Skalierung: Ein Modell würde die ID 1000 als „100-mal bedeutender“ als die ID 10 behandeln, was für Sprache keinen Sinn ergibt.

Stattdessen verwenden wir Embeddings.

**Was sind Embeddings?**

Ein Embedding ersetzt eine einzelne ID durch einen Vektor (eine Liste von Zahlen).
Anstelle einer einzelnen Zahl nutzen wir mehrere Dimensionen (z. B. 256, 768 oder 1536), um ein Token zu beschreiben. Jede Dimension kann theoretisch ein anderes Merkmal des Wortes lernen – wie seine grammatikalische Rolle, seine Stimmung (Sentiment) oder seine Beziehung zu anderen Konzepten. Dies ermöglicht es dem Modell, Ähnlichkeiten zu berechnen: Wörter wie „Katze“ und „Hund“ werden letztendlich ähnliche Vektoren aufweisen und in diesem hochdimensionalen Raum nah beieinander liegen.

Diese Embeddings werden in einer großen Embedding-Matrix zusammengefasst.

**Embedding-Matrix**

Die Embedding-Matrix ist im Grunde eine riesige Nachschlagetabelle.
- Zeilen: Die Anzahl der Zeilen entspricht der Vokabulargröße (Vocabulary Size, z. B. 100.277 für cl100k_base). Dies stellt sicher, dass jedes mögliche Token genau einen Eintrag hat.
- Spalten: Die Anzahl der Spalten ist die Embedding-Dimension (z. B. 768).

Wie es funktioniert: Wenn Sie eine Token-ID an die Schicht übergeben, ruft sie einfach die Zeile an diesem Index ab. Die ID 1 entspricht der Zeile 1. Es gibt keine Berechnung – nur ein schnelles Nachschlagen.

---

**Beispiel**

Um dies zu veranschaulichen, verwenden wir eine winzige $6 \times 3$-Matrix. Das bedeutet, dass das Modell nur 6 einzigartige Token kennt und jedes Token durch nur 3 Dimensionen beschrieben wird. Zu Beginn des Trainings sind diese Zahlen zufällig und haben keine Bedeutung – das Modell lernt die „Bedeutung“ erst durch das Training.

In [185]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [186]:
# Konvertiert die Token-ID 3 in den entsprechenden Embedding-Vektor
# Unsere Token-ID ist 3, daher schlagen wir in der 3. Zeile der Gewichtsmatrix der Embedding-Schicht nach
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [187]:
# Jedes Token einbetten (Embedding)
# Diese transformierte Matrix dient nun als numerische Eingabe für die nachfolgenden Schichten des neuronalen Netzwerks während des Trainings.
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## 2.8 Kodierung von Wortpositionen

Zu diesem Zeitpunkt haben wir unsere Token-IDs erfolgreich in dichte Vektoren transformiert. Es gibt jedoch ein grundlegendes Problem: **Token-Embeddings sind positionsinvariant.**

Die eingebetteten Vektoren der Sätze „Der Hund beißt den Mann“ und „Der Mann beißt den Hund“ sind in beiden Fällen identisch. Ohne zusätzliche Informationen würde ein Transformer-Modell diese Sätze wie eine ungeordnete Wortsammlung („Bag of Words“) behandeln, ohne sich der Reihenfolge bewusst zu sein. Um dies zu beheben, verwenden wir **Positional Embeddings** (Positions-Einbettungen).

Wir erstellen eine zweite Embedding-Matrix, die nicht die „Bedeutung von Wörtern“, sondern vielmehr die „Bedeutung von Positionen“ speichert (z. B. einen Vektor für „Position 0“, einen Vektor für „Position 1“).

Um diese beiden Informationen zu kombinieren, addieren wir einfach den Token-Embedding-Vektor und den Positional-Embedding-Vektor zusammen. Der resultierende Vektor enthält sowohl die semantische Bedeutung des Wortes als auch seine Position in der Sequenz.

In [188]:
# Der BytePair-Encoder cl100k_base hat eine Vokabulargröße von 100.277:
# Angenommen, wir möchten die Eingabe-Token in eine 256-dimensionale Vektordarstellung kodieren:
vocab_size = 100277
output_dim = 256

# 1.) Initialisieren der Embedding-Schicht mit der angegebenen Vokabulargröße und Ausgabedimension
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [189]:
# Extrahiert den ersten Batch von Token-IDs, jedoch mit anderen Parametern für den DataLoader
max_length = 4
batch_size = 8

dataloader = create_dataloader_v1(
    raw_text, batch_size=batch_size, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)

inputs, targets = next(data_iter)
print("Token IDs (erste 2 Sequenzen):\n", inputs[:2])
print("\nTargets (erste 2 Sequenzen):\n", targets[:2])
print(f"\nInputs shape: {inputs.shape}")

token_embeddings = token_embedding_layer(inputs)
print(f"Token embeddings shape: {token_embeddings.shape}")
print("Embedding-Ausschnitt [Batch 0, Token 0, erste 8 Dimensionen]:")
print(token_embeddings[0, 0, :8])

Token IDs (erste 2 Sequenzen):
 tensor([[29880,    25,  2360,  7826],
        [  731, 18878, 27085,   198]])

Targets (erste 2 Sequenzen):
 tensor([[   25,  2360,  7826,   731],
        [18878, 27085,   198, 46847]])

Inputs shape: torch.Size([8, 4])
Token embeddings shape: torch.Size([8, 4, 256])
Embedding-Ausschnitt [Batch 0, Token 0, erste 8 Dimensionen]:
tensor([ 0.3908,  0.5987,  0.7033, -0.5252, -0.1755,  0.0307,  0.2410,  0.7676],
       grad_fn=<SliceBackward0>)


In [190]:
# Zweites Beispiel mit kleinerem Kontextfenster
max_length = 3
batch_size = 3

dataloader = create_dataloader_v1(
    raw_text, batch_size=batch_size, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)

inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print(f"\nInputs shape: {inputs.shape}")

token_embeddings = token_embedding_layer(inputs)
print(f"Token embeddings shape: {token_embeddings.shape}")
print("Embedding-Ausschnitt [Batch 0, alle Tokens, erste 8 Dimensionen]:")
print(token_embeddings[0, :, :8])

Token IDs:
 tensor([[29880,    25,  2360],
        [ 7826,   731, 18878],
        [27085,   198, 46847]])

Inputs shape: torch.Size([3, 3])
Token embeddings shape: torch.Size([3, 3, 256])
Embedding-Ausschnitt [Batch 0, alle Tokens, erste 8 Dimensionen]:
tensor([[ 0.3908,  0.5987,  0.7033, -0.5252, -0.1755,  0.0307,  0.2410,  0.7676],
        [-0.0847,  1.5248,  0.1663,  0.1113, -0.5908,  1.2230, -0.0483, -0.3928],
        [-0.5454, -0.2515, -1.5502, -0.6143,  0.9544, -1.2116, -0.1025,  0.9482]],
       grad_fn=<SliceBackward0>)


In [191]:
# Erstellt eine Positional-Embedding-Schicht für die Kontextlänge des Modells
# Die Kontextlänge ist die maximale Anzahl von Tokens, die das Modell in einem Forward-Pass verarbeiten kann.
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

print(f"Positional embedding weight shape: {pos_embedding_layer.weight.shape}")
print("Erste Positionsvektoren (erste 8 Dimensionen):")
print(pos_embedding_layer.weight[:min(2, context_length), :8])

Positional embedding weight shape: torch.Size([3, 256])
Erste Positionsvektoren (erste 8 Dimensionen):
tensor([[ 0.7770, -0.2792,  1.4069, -0.0382, -1.8392,  0.6663,  0.7372, -0.1367],
        [ 1.1292,  1.3358,  0.7712,  1.5759, -1.2474,  0.7461,  0.7900,  0.9863]],
       grad_fn=<SliceBackward0>)


In [192]:
# Erstellt einen einfachen Tensor aus Positionsindizes.
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(f"Positional embeddings shape: {pos_embeddings.shape}")
print("Positions-Embedding-Ausschnitt (erste 8 Dimensionen):")
print(pos_embeddings[:, :8])

Positional embeddings shape: torch.Size([3, 256])
Positions-Embedding-Ausschnitt (erste 8 Dimensionen):
tensor([[ 0.7770, -0.2792,  1.4069, -0.0382, -1.8392,  0.6663,  0.7372, -0.1367],
        [ 1.1292,  1.3358,  0.7712,  1.5759, -1.2474,  0.7461,  0.7900,  0.9863],
        [ 0.2338,  0.8553, -1.1240, -0.3834,  0.5923,  1.1351,  0.7119,  0.3513]],
       grad_fn=<SliceBackward0>)


In [193]:
# Um nun die finalen Input-Embeddings zu erstellen, addieren wir Token- und Positional-Embeddings.
input_embeddings = token_embeddings + pos_embeddings
print(f"Input embeddings shape: {input_embeddings.shape}")
print("Input-Embedding-Ausschnitt [Batch 0, alle Tokens, erste 8 Dimensionen]:")
print(input_embeddings[0, :, :8])

Input embeddings shape: torch.Size([3, 3, 256])
Input-Embedding-Ausschnitt [Batch 0, alle Tokens, erste 8 Dimensionen]:
tensor([[ 1.1678,  0.3195,  2.1102, -0.5634, -2.0147,  0.6970,  0.9782,  0.6309],
        [ 1.0445,  2.8606,  0.9375,  1.6872, -1.8381,  1.9692,  0.7418,  0.5935],
        [-0.3116,  0.6038, -2.6742, -0.9977,  1.5467, -0.0765,  0.6094,  1.2995]],
       grad_fn=<SliceBackward0>)


## Kurze Zusammenfassung

Wir haben rohen Text erfolgreich in ein Input-Embedding transformiert – ein Format, das ein Large Language Model „verstehen“ und verarbeiten kann.

Um das finale Input-Embedding zu erhalten, haben wir zwei verschiedene Schichten kombiniert:

- Token-Embeddings: Konvertieren Token-IDs in 256-dimensionale Vektoren, um die semantische Bedeutung zu erfassen.
- Positional-Embeddings: Fügen für jede Position einzigartige Vektoren hinzu, um die Wortreihenfolge zu erfassen.

Der resultierende input_embeddings-Tensor ist nun bereit für das neuronale Netzwerk.